# EE/CS 148B — HW4: Diffusion Models

**Open this notebook in Colab via:** File → Open notebook → GitHub → `trevorbchen/cs148_hw4`

Runtime: **A100 GPU** (Runtime → Change runtime type → A100)

## 0. Setup

In [ ]:
# Clone repo and install dependencies
import os

REPO = 'https://github.com/trevorbchen/cs148_hw4'
ROOT = '/content/cs148_hw4'

if not os.path.isdir(ROOT):
    !git clone {REPO} {ROOT}
else:
    # Pull latest changes if already cloned
    !git -C {ROOT} pull

%cd {ROOT}
!pip install -q -e .
!pip install -q torch-fidelity

In [ ]:
import sys, torch
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs('figures', exist_ok=True)

In [ ]:
# Sanity check — run autograder tests
!python -m pytest tests/ -v --tb=short 2>&1 | tail -20

---
## Part 1.8 — DDPM Coefficient Plot

In [ ]:
import torch, matplotlib.pyplot as plt

T = 1000
beta_min, beta_max = 0.01, 5.0
ts = torch.arange(1, T + 1).float()
beta_t = beta_min / T + (beta_max / T - beta_min / T) * ts
alpha_t = 1.0 - beta_t
alpha_bar = torch.cumprod(alpha_t, dim=0)
coeff = beta_t**2 / (2 * beta_t * alpha_t * (1 - alpha_bar))

plt.figure(figsize=(8, 4))
plt.semilogy(ts.numpy(), coeff.numpy())
plt.xlabel('t')
plt.ylabel('Coefficient (log scale)')
plt.title(r'DDPM loss coefficient $\beta_t^2 / (2\sigma_t^2 \alpha_t (1-\bar{\alpha}_t))$')
plt.tight_layout()
plt.savefig('figures/ddpm_coefficient_plot.png', dpi=150)
plt.show()

---
## Part 5 — VP-SDE Score Model

### 5.C.i — FashionMNIST Dataset Visualization

In [ ]:
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torchvision.utils import make_grid

FASHION_CLASSES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot',
]

tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
train_ds = datasets.FashionMNIST('data', train=True, download=True, transform=tf)

imgs = torch.stack([train_ds[i][0] for i in range(64)])
grid = make_grid(imgs * 0.5 + 0.5, nrow=8)

plt.figure(figsize=(10, 10))
plt.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
plt.title('FashionMNIST — 64 training samples (28×28 grayscale, 10 classes)')
plt.axis('off')
plt.tight_layout()
plt.savefig('figures/fashionmnist_samples.png', dpi=150)
plt.show()
print('Classes:', FASHION_CLASSES)
print('Image shape:', train_ds[0][0].shape)

### 5.C.ii — Train VP Score Model (~10 min on A100)

In [ ]:
!python scripts/train_vp.py \
    --beta_min 0.01 --beta_max 5.0 \
    --epochs 50 --lr 1e-4 --batch_size 128 \
    --patience 10 \
    --save_dir runs/vp \
    --device {DEVICE}

In [ ]:
import numpy as np

train_losses = np.load('runs/vp/train_losses.npy')
val_losses   = np.load('runs/vp/val_losses.npy')

plt.figure(figsize=(8, 4))
plt.semilogy(train_losses, label='train')
plt.semilogy(val_losses,   label='val')
plt.xlabel('Epoch')
plt.ylabel('Loss (log scale)')
plt.title('VP Score Model — Training Curves (β_min=0.01, β_max=5.0)')
plt.legend()
plt.tight_layout()
plt.savefig('figures/vp_training_curves.png', dpi=150)
plt.show()

### 5.C.iii — EM Samples

In [ ]:
!python scripts/sample.py \
    --method em \
    --checkpoint runs/vp/best.pt \
    --beta_min 0.01 --beta_max 5.0 \
    --num_steps 1000 --n_samples 64 \
    --out figures/vp_em_samples.png \
    --device {DEVICE}

from IPython.display import Image
Image('figures/vp_em_samples.png')

### 5.C.iv — PC Samples (corrector steps = 1 and 3)

In [ ]:
for n_corr in [1, 3]:
    !python scripts/sample.py \
        --method pc \
        --checkpoint runs/vp/best.pt \
        --beta_min 0.01 --beta_max 5.0 \
        --num_steps 1000 --n_corrector {n_corr} \
        --n_samples 64 \
        --out figures/vp_pc_{n_corr}corr_samples.png \
        --device {DEVICE}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, n_corr in zip(axes, [1, 3]):
    img = plt.imread(f'figures/vp_pc_{n_corr}corr_samples.png')
    ax.imshow(img, cmap='gray')
    ax.set_title(f'PC sampler — {n_corr} corrector step(s)')
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## Part 6 — Rectified Flow

### 6.A — Train Rectified Flow + Compare Loss Curves

In [ ]:
!python scripts/train_rectflow.py \
    --epochs 50 --lr 1e-4 --batch_size 128 \
    --save_dir runs/rectflow \
    --device {DEVICE}

In [ ]:
vp_train = np.load('runs/vp/train_losses.npy')
rf_train = np.load('runs/rectflow/train_losses.npy')

plt.figure(figsize=(8, 4))
plt.semilogy(vp_train, label='VP Score Model')
plt.semilogy(rf_train,  label='Rectified Flow')
plt.xlabel('Epoch')
plt.ylabel('Loss (log scale)')
plt.title('Training Loss Comparison — VP vs Rectified Flow')
plt.legend()
plt.tight_layout()
plt.savefig('figures/combined_training_curves.png', dpi=150)
plt.show()

### 6.B — Euler Sampler at Multiple Step Counts

In [ ]:
for steps in [1, 5, 10, 50, 100, 200]:
    !python scripts/sample.py \
        --method rectflow \
        --checkpoint runs/rectflow/best.pt \
        --num_steps {steps} --n_samples 64 \
        --out figures/rf_{steps}steps.png \
        --device {DEVICE}

In [ ]:
# KID evaluation table (takes a few minutes)
!python scripts/eval_kid.py \
    --vp_checkpoint  runs/vp/best.pt \
    --rf_checkpoint  runs/rectflow/best.pt \
    --beta_min 0.01 --beta_max 5.0 \
    --n_samples 1000 \
    --device {DEVICE}

### 6.C — Reflow: One-Step Generation

In [ ]:
!python scripts/train_rectflow.py \
    --reflow \
    --checkpoint runs/rectflow/best.pt \
    --n_reflow_pairs 50000 --reflow_steps 100 \
    --epochs 20 --lr 1e-4 --batch_size 128 \
    --save_dir runs/rectflow_reflow \
    --device {DEVICE}

In [ ]:
!python scripts/sample.py \
    --method rectflow \
    --checkpoint runs/rectflow_reflow/best.pt \
    --num_steps 1 --n_samples 64 \
    --out figures/reflow_1step.png \
    --device {DEVICE}

Image('figures/reflow_1step.png')

### 6.D — Side-by-Side Qualitative Comparison Grid

In [ ]:
!python scripts/sample.py \
    --method all \
    --vp_checkpoint      runs/vp/best.pt \
    --rf_checkpoint      runs/rectflow/best.pt \
    --reflow_checkpoint  runs/rectflow_reflow/best.pt \
    --beta_min 0.01 --beta_max 5.0 \
    --n_samples 8 --seed 42 \
    --out figures/comparison_grid.png \
    --device {DEVICE}

Image('figures/comparison_grid.png')

---
## Part 7 — Guided Diffusion (256×256)

In [ ]:
# Clone guided-diffusion and download weights (~2 GB, run once)
if not os.path.isdir('guided-diffusion'):
    !git clone https://github.com/openai/guided-diffusion.git

!pip install -q mpi4py blobfile
!pip install -q -e guided-diffusion/

os.makedirs('guided-diffusion/models', exist_ok=True)

!wget -q -nc -P guided-diffusion/models \
    https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt
!wget -q -nc -P guided-diffusion/models \
    https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_classifier.pt

!ls -lh guided-diffusion/models/

In [ ]:
import sys
sys.path.insert(0, os.path.abspath('guided-diffusion'))
import numpy as np, glob
from torchvision.utils import make_grid
from guided_diffusion import dist_util
from guided_diffusion.script_util import (
    model_and_diffusion_defaults, create_model_and_diffusion,
)

os.makedirs('outputs/guided', exist_ok=True)

# Shared model config
UNCOND_DEFAULTS = dict(
    attention_resolutions='32,16,8', class_cond=False,
    diffusion_steps=1000, image_size=256, learn_sigma=True,
    noise_schedule='linear', num_channels=256, num_head_channels=64,
    num_res_blocks=2, resblock_updown=True,
    use_fp16=True, use_scale_shift_norm=True,
)

def load_uncond_model():
    cfg = {**model_and_diffusion_defaults(), **UNCOND_DEFAULTS}
    model, diffusion = create_model_and_diffusion(**cfg)
    model.load_state_dict(
        dist_util.load_state_dict(
            'guided-diffusion/models/256x256_diffusion_uncond.pt', map_location='cpu'
        )
    )
    model.to(DEVICE)
    model.convert_to_fp16()
    model.eval()
    return model, diffusion

model, diffusion = load_uncond_model()
print('Model loaded')

### 7.1 — Unconditional Generation

In [ ]:
torch.manual_seed(0)
samples = diffusion.p_sample_loop(
    model, (8, 3, 256, 256),
    clip_denoised=True, model_kwargs={}, progress=True,
)

grid = make_grid(samples.float() * 0.5 + 0.5, nrow=8)
plt.figure(figsize=(16, 2))
plt.imshow(grid.permute(1, 2, 0).cpu().clamp(0, 1).numpy())
plt.title('Unconditional 256×256 generation (8 samples)')
plt.axis('off')
plt.tight_layout()
plt.savefig('figures/guided_uncond.png', dpi=100, bbox_inches='tight')
plt.show()

### 7.2 — Progressive Generation

In [ ]:
torch.manual_seed(1)
noise = torch.randn(1, 3, 256, 256, device=DEVICE)

n_snap, intermediates = 8, []
all_steps = list(diffusion.p_sample_loop_progressive(
    model, (1, 3, 256, 256), noise=noise.clone(),
    clip_denoised=True, model_kwargs={},
))
indices = [int(i * (len(all_steps) - 1) / (n_snap - 1)) for i in range(n_snap)]
frames = torch.cat([all_steps[i]['sample'].cpu().float() for i in indices])

grid = make_grid(frames * 0.5 + 0.5, nrow=n_snap)
plt.figure(figsize=(16, 2))
plt.imshow(grid.permute(1, 2, 0).clamp(0, 1).numpy())
plt.title('Progressive generation: noise (left) → image (right)')
plt.axis('off')
plt.tight_layout()
plt.savefig('figures/guided_progressive.png', dpi=100, bbox_inches='tight')
plt.show()

### 7.3 — Noise Interpolation

In [ ]:
torch.manual_seed(0); z0 = torch.randn(1, 3, 256, 256, device=DEVICE)
torch.manual_seed(7); z7 = torch.randn(1, 3, 256, 256, device=DEVICE)

interp_samples = []
for i in range(8):
    alpha = i / 7.0
    zi = (1 - alpha) * z0 + alpha * z7
    s = diffusion.p_sample_loop(
        model, (1, 3, 256, 256), noise=zi.clone(),
        clip_denoised=True, model_kwargs={},
    )
    interp_samples.append(s.cpu().float())
    print(f'  {i}/7 done')

strip = torch.cat(interp_samples)
grid = make_grid(strip * 0.5 + 0.5, nrow=8)
plt.figure(figsize=(16, 2))
plt.imshow(grid.permute(1, 2, 0).clamp(0, 1).numpy())
plt.title('Noise interpolation: z0 (left) → z7 (right)')
plt.axis('off')
plt.tight_layout()
plt.savefig('figures/guided_interpolation.png', dpi=100, bbox_inches='tight')
plt.show()

### 7.4 — Conditional Generation (classifier guidance)

In [ ]:
from guided_diffusion.script_util import classifier_defaults, create_classifier

clf_cfg = {
    'image_size': 256,
    'classifier_use_fp16': False,
    'classifier_width': 128,
    'classifier_depth': 2,
    'classifier_attention_resolutions': '32,16,8',
    'classifier_use_scale_shift_norm': True,
    'classifier_resblock_updown': True,
    'classifier_pool': 'attention',
}
classifier = create_classifier(**clf_cfg)
classifier.load_state_dict(
    dist_util.load_state_dict(
        'guided-diffusion/models/256x256_classifier.pt', map_location='cpu'
    )
)
classifier.to(DEVICE).eval()
print('Classifier loaded')

import random; random.seed(42)
classes = random.sample(range(1000), 8)
print('Target classes:', classes)

SCALE = 1.0
# model_kwargs must be empty: the unconditional model asserts y is None.
# p_sample_loop passes model_kwargs to both model() and cond_fn(), so we
# capture the class targets via closure instead.
_y = torch.tensor(classes, device=DEVICE)

def cond_fn(x, t):
    with torch.enable_grad():
        x_in = x.detach().float().requires_grad_(True)
        logits = classifier(x_in, t)
        log_p = torch.nn.functional.log_softmax(logits, dim=-1)
        sel = log_p[range(len(logits)), _y]
        return torch.autograd.grad(sel.sum(), x_in)[0] * SCALE

torch.manual_seed(42)
cond_samples = diffusion.p_sample_loop(
    model, (8, 3, 256, 256),
    clip_denoised=True, model_kwargs={},
    cond_fn=cond_fn, progress=True,
)

grid = make_grid(cond_samples.float() * 0.5 + 0.5, nrow=8)
plt.figure(figsize=(16, 2))
plt.imshow(grid.permute(1, 2, 0).cpu().clamp(0, 1).numpy())
plt.title(f'Conditional generation (classifier guidance) — classes: {classes}')
plt.axis('off')
plt.tight_layout()
plt.savefig('figures/guided_conditional.png', dpi=100, bbox_inches='tight')
plt.show()

### 7.5 — Classifier Scale Sweep

In [ ]:
scales = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0, 7.5, 10.0]
target_class = 291   # lion
torch.manual_seed(42)
fixed_noise = torch.randn(2, 3, 256, 256, device=DEVICE)
_y_sweep = torch.tensor([target_class, target_class], device=DEVICE)

rows = []
for scale in scales:
    print(f'  scale={scale}', end='', flush=True)

    def cond_fn(x, t, _scale=scale):
        with torch.enable_grad():
            x_in = x.detach().float().requires_grad_(True)
            logits = classifier(x_in, t)
            log_p = torch.nn.functional.log_softmax(logits, dim=-1)
            sel = log_p[range(len(logits)), _y_sweep]
            return torch.autograd.grad(sel.sum(), x_in)[0] * _scale

    s = diffusion.p_sample_loop(
        model, (2, 3, 256, 256), noise=fixed_noise.clone(),
        clip_denoised=True, model_kwargs={}, cond_fn=cond_fn,
    )
    rows.append(s.cpu().float())
    print(' done')

row1 = torch.cat([r[0:1] for r in rows])
row2 = torch.cat([r[1:2] for r in rows])
combined = torch.cat([row1, row2])

grid = make_grid(combined * 0.5 + 0.5, nrow=8)
plt.figure(figsize=(16, 4))
plt.imshow(grid.permute(1, 2, 0).clamp(0, 1).numpy())
plt.title(f'Classifier scale sweep: {scales}\n(2 rows, class={target_class}=lion)')
plt.axis('off')
plt.tight_layout()
plt.savefig('figures/guided_classifier_sweep.png', dpi=100, bbox_inches='tight')
plt.show()